# HPO single_view_complex_v2 (v2) - inherited head configuration

This variant does **not** run its own search. It inherits the head configuration
of `dual_view_v2` (`config.HPO_INHERITS`), because its head is architecturally identical to the dual-view head - the same CNN,
BiLSTM and classifier - and only the second view and the cross-attention are
removed.

Inheriting is not a shortcut here, it is what keeps the comparison against
`dual_view_v2` single-variable: both run the identical head configuration, so
the only thing that differs between them is the architectural change itself.

Architectures whose head differs - `baseline_v2` and `dual_view_v2` - each run
their own search over the same space and with the same budget
(`config.HPO_SEARCHES`).

**Output:** `hpo/single_view_complex_v2/best_params.json` (no training happens here)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json
sys.path.insert(0, "/content/drive/MyDrive/google_colab/kusa/v2_heldout")
from config import *
import utils_split as u

In [ ]:
VARIANT = "single_view_complex_v2"
SOURCE  = HPO_INHERITS[VARIANT]          # "dual_view_v2"

SRC_PATH = best_params_path(SOURCE)
assert os.path.exists(SRC_PATH), (
    f"{SRC_PATH} is missing - run hpo_kusa_dual_view_v2 first.")
with open(SRC_PATH, encoding="utf-8") as f:
    src = json.load(f)

print(f"{SOURCE} best params (source of truth):")
for k, v in src.items():
    if not k.startswith("_"):
        print(f"  {k:14s} = {v}")
print(f"  (best HPO-slice value: {src['_best_value']:.4f}, {src['_n_trials']} trials)")

In [ ]:
# The CV notebook reads: batch_size, encoder_lr, head_lr, dropout,
# weight_decay, warmup_ratio, epochs.
best = {
    "batch_size":   src["batch_size"],
    "encoder_lr":   src["encoder_lr"],
    "head_lr":      src["head_lr"],
    "dropout":      src["dropout"],
    "weight_decay": src["weight_decay"],
    "warmup_ratio": src["warmup_ratio"],
    "epochs":       src["epochs"],
    "_variant":          VARIANT,
    "_inherited_from":   SOURCE,
    "_note":             ("identical head architecture; inheriting keeps the "
                          "comparison against the source single-variable"),
    "_best_value":       src["_best_value"],
    "_n_trials":         0,
    "_hpo_slice_seed":   src.get("_hpo_slice_seed"),
    "_hpo_seed":         src.get("_hpo_seed"),
}

out = best_params_path(VARIANT)
with open(out, "w", encoding="utf-8") as f:
    json.dump(best, f, indent=2, ensure_ascii=False)
print(json.dumps(best, indent=2, ensure_ascii=False))
print("\nsaved:", out)

assert_test_untouched(globals())